# SHBT261 TextVQA Full Prompt-Engineering Pipeline

This notebook runs the full 5,000-example TextVQA validation experiment using Qwen2.5-VL-3B-Instruct and four prompt conditions: plain, concise, OCR-aware, and strict OCR-aware. It saves all predictions, metrics, report tables, plots, and qualitative examples inside this uploaded `textvqa_final` folder.


In [ ]:
# Install dependencies. Runtime restart is usually not needed after this cell.
import sys, subprocess
packages = [
    'transformers>=4.51.0', 'accelerate>=0.34.0', 'bitsandbytes>=0.43.3',
    'datasets>=2.20.0', 'qwen-vl-utils[decord]', 'nltk', 'matplotlib',
    'seaborn', 'pandas', 'tabulate',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *packages])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Upload the folder directly to: MyDrive/textvqa_final
PROJECT_DIR = '/content/drive/MyDrive/textvqa_final'
CODE_DIR = PROJECT_DIR

import os, sys
os.makedirs(PROJECT_DIR, exist_ok=True)
sys.path.insert(0, CODE_DIR)

print('Project dir:', PROJECT_DIR)
print('Code dir:', CODE_DIR)
print('Files:', sorted(os.listdir(CODE_DIR)))


In [ ]:
import importlib
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

import textvqa_pipeline
importlib.reload(textvqa_pipeline)
from textvqa_pipeline import *

base_config = RunConfig(
    project_dir=PROJECT_DIR,
    max_new_tokens=8,
    inference_batch_size=4,
)

set_seed(base_config.seed)
preflight()
ensure_dirs(PROJECT_DIR)


## Full Validation Prompt Ablation

This cell runs all four prompt styles on the full 5,000-example validation split. It writes partial predictions every 250 samples and final predictions/metrics for each prompt to `textvqa_final/results/`.


In [ ]:
PROMPT_STYLES = ['plain', 'concise', 'ocr', 'strict_ocr']
VAL_N = None  # None means the full 5,000-example validation split

model, processor = get_model_and_processor(base_config)
val_ds = load_textvqa_split('validation', max_samples=VAL_N)

prompt_runs = {}
for style in PROMPT_STYLES:
    prefix = f'prompt_{style}_valfull'
    results, metrics = run_inference(
        model, processor, val_ds, base_config,
        output_prefix=prefix,
        prompt_style=style,
        max_samples=VAL_N,
    )
    prompt_runs[style] = {'prefix': prefix, 'metrics': metrics}

prompt_runs


## Report Assets

This cell creates the report-ready outputs: summary tables, LaTeX tables, category heatmaps, metric comparison plots, error analysis, OCR-count and answer-length diagnostics, transition plots, and qualitative examples.


In [ ]:
from pathlib import Path

results_dir = Path(PROJECT_DIR) / 'results'
prompt_prefixes = [prompt_runs[style]['prefix'] for style in PROMPT_STYLES]

for prefix in prompt_prefixes:
    analyze_predictions(results_dir / f'{prefix}_predictions.json', prefix, PROJECT_DIR)

summary = summarize_prompt_runs(prompt_prefixes, PROJECT_DIR, output_name='full_prompt_ablation_summary')
engineered_styles = ['concise', 'ocr', 'strict_ocr']
best_engineered_style = max(engineered_styles, key=lambda s: prompt_runs[s]['metrics']['accuracy'])
best_engineered_prefix = prompt_runs[best_engineered_style]['prefix']

compare_prediction_files(
    results_dir / 'prompt_plain_valfull_predictions.json',
    results_dir / f'{best_engineered_prefix}_predictions.json',
    'plain_baseline', f'best_engineered_{best_engineered_style}', PROJECT_DIR,
)
compare_prediction_files(
    results_dir / 'prompt_concise_valfull_predictions.json',
    results_dir / f'{best_engineered_prefix}_predictions.json',
    'concise_baseline', f'best_engineered_{best_engineered_style}', PROJECT_DIR,
)

make_plots([results_dir / f'{prefix}_metrics.json' for prefix in prompt_prefixes], PROJECT_DIR)
report_assets = create_report_assets(
    prompt_prefixes,
    PROJECT_DIR,
    baseline_prefix='prompt_concise_valfull',
    best_prefix=best_engineered_prefix,
    output_name='full_prompt_report_assets',
)

print('Best engineered prompt:', best_engineered_style)
print('Summary:', results_dir / 'full_prompt_ablation_summary.csv')
print('Report tables:', results_dir / 'report_tables')
print('Report examples:', results_dir / 'report_examples')
print('Figures:', Path(PROJECT_DIR) / 'figures')
report_assets


In [ ]:
import importlib, sys
from pathlib import Path
from collections import Counter

PROJECT_DIR = "/content/drive/MyDrive/textvqa_final"
CODE_DIR = PROJECT_DIR

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import textvqa_pipeline
importlib.reload(textvqa_pipeline)
from textvqa_pipeline import *

prompt_prefixes = [
    "prompt_plain_valfull",
    "prompt_concise_valfull",
    "prompt_ocr_valfull",
    "prompt_strict_ocr_valfull",
]

# Fast sanity check: estimates how many examples need the LLM judge.
for prefix in prompt_prefixes:
    rows = load_json(Path(PROJECT_DIR) / "results" / f"{prefix}_predictions.json")
    counts = Counter()
    for row in rows:
        if textvqa_accuracy(row.get("prediction", ""), row.get("ground_truths", [])) > 0:
            counts["AUTO_EXACT_MATCH"] += 1
        else:
            score, reason = strict_judge_prefilter(
                row.get("question", ""),
                row.get("prediction", ""),
                row.get("ground_truths", []),
            )
            counts["LLM_CALL" if score is None else reason] += 1
    print(prefix, dict(counts))


In [ ]:
strict_judge_summary = judge_predictions_with_llm(
    prefixes=prompt_prefixes,
    project_dir=PROJECT_DIR,
    judge_model_name="Qwen/Qwen2.5-7B-Instruct",
    batch_size=16,
    output_name="llm_judge_similarity_strict7b",
    use_strict_prefilter=True,
    resume=True,
    checkpoint_every_batches=25,
)

strict_judge_summary
